In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Create a sample movie ratings dataset
np.random.seed(42)

movies = {
    1: 'The Dark Knight', 2: 'Inception', 3: 'Interstellar',
    4: 'The Matrix', 5: 'Avengers', 6: 'Titanic',
    7: 'The Godfather', 8: 'Forrest Gump', 9: 'Parasite', 10: 'Joker'
}

users = ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank', 'Grace', 'Henry']

# Ratings matrix (0 = not rated)
ratings_data = {
    'Alice':  [5, 4, 5, 4, 0, 0, 3, 0, 4, 5],
    'Bob':    [4, 0, 4, 5, 3, 0, 0, 4, 0, 4],
    'Carol':  [0, 5, 4, 0, 0, 5, 4, 5, 3, 0],
    'David':  [5, 4, 0, 4, 5, 0, 0, 0, 5, 4],
    'Eve':    [0, 4, 5, 0, 0, 4, 5, 4, 4, 0],
    'Frank':  [4, 0, 0, 5, 4, 0, 3, 0, 0, 5],
    'Grace':  [0, 3, 4, 0, 0, 5, 4, 5, 4, 0],
    'Henry':  [5, 5, 4, 4, 0, 0, 0, 3, 5, 4],
}

df = pd.DataFrame(ratings_data, index=movies.values())
print("Ratings Matrix:")
print(df)

Ratings Matrix:
                 Alice  Bob  Carol  David  Eve  Frank  Grace  Henry
The Dark Knight      5    4      0      5    0      4      0      5
Inception            4    0      5      4    4      0      3      5
Interstellar         5    4      4      0    5      0      4      4
The Matrix           4    5      0      4    0      5      0      4
Avengers             0    3      0      5    0      4      0      0
Titanic              0    0      5      0    4      0      5      0
The Godfather        3    0      4      0    5      3      4      0
Forrest Gump         0    4      5      0    4      0      5      3
Parasite             4    0      3      5    4      0      4      5
Joker                5    4      0      4    0      5      0      4


In [2]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute user similarity
user_similarity = cosine_similarity(df.T)
user_sim_df = pd.DataFrame(user_similarity, index=df.columns, columns=df.columns)

print("User Similarity Matrix:")
print(user_sim_df.round(2))

# Recommendation function
def recommend_movies(user, n=3):
    # Get similar users
    sim_users = user_sim_df[user].drop(user).sort_values(ascending=False)
    
    # Movies not yet rated by this user
    unrated = df[df[user] == 0].index.tolist()
    
    # Score unrated movies based on similar users' ratings
    scores = {}
    for movie in unrated:
        score = 0
        total_sim = 0
        for sim_user, sim_score in sim_users.items():
            rating = df.loc[movie, sim_user]
            if rating > 0:
                score += sim_score * rating
                total_sim += sim_score
        if total_sim > 0:
            scores[movie] = score / total_sim
    
    # Return top N recommendations
    recommended = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]
    print(f"\n🎬 Top {n} movie recommendations for {user}:")
    for i, (movie, score) in enumerate(recommended, 1):
        print(f"  {i}. {movie} (predicted rating: {score:.2f})")

# Test for different users
recommend_movies('Alice')
recommend_movies('Bob')
recommend_movies('Carol')

User Similarity Matrix:
       Alice   Bob  Carol  David   Eve  Frank  Grace  Henry
Alice   1.00  0.70   0.52   0.76  0.59   0.68   0.50   0.92
Bob     0.70  1.00   0.34   0.65  0.34   0.77   0.35   0.74
Carol   0.52  0.34   1.00   0.29  0.97   0.12   0.98   0.57
David   0.76  0.65   0.29   1.00  0.30   0.76   0.28   0.80
Eve     0.59  0.34   0.97   0.30  1.00   0.15   0.98   0.59
Frank   0.68  0.77   0.12   0.76  0.15   1.00   0.12   0.55
Grace   0.50  0.35   0.98   0.28  0.98   0.12   1.00   0.56
Henry   0.92  0.74   0.57   0.80  0.59   0.55   0.56   1.00

🎬 Top 3 movie recommendations for Alice:
  1. Titanic (predicted rating: 4.64)
  2. Forrest Gump (predicted rating: 4.03)
  3. Avengers (predicted rating: 4.03)

🎬 Top 3 movie recommendations for Bob:
  1. Titanic (predicted rating: 4.67)
  2. Parasite (predicted rating: 4.34)
  3. Inception (predicted rating: 4.23)

🎬 Top 3 movie recommendations for Carol:
  1. The Dark Knight (predicted rating: 4.75)
  2. Joker (predicted rating: